In [1]:
"""
Step 1: Get the topological order of DAGs.
Step 2: Generate the scenario.
Step 3: Run the three algorithms and compare the results.
"""
import datetime

from models.scheduler.dpe import DPE
from models.scheduler.heft import HEFT
from models.utils.dataset import *
from models.utils.figure import *
from models.utils.scenario import *

In [2]:
# 采集数据，并对 task 进行拓扑排序
sample_jobs()
get_topological_order()

Dataset (batch_task.csv & batch_instance.csv) is already selected.
Jobs' topological order has been obtained.


In [3]:
# 生成边缘计算场景
G, bw, pp = generate_scenario()
print_scenario(G, bw, pp)
simple_paths = get_simple_paths(G)
print_simple_paths(simple_paths)
reciprocals_list, proportions_list = get_ratio(simple_paths, bw)
pp_required, data_stream = set_funcs()


The connected graph of edge servers (represented by adjacent matrix):
array([[1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.],
       [1., 1., 1., 1.]])

====> throughput of each link <====
array([[-1., 37., 51., 56.],
       [37., -1., 68., 59.],
       [51., 68., -1., 66.],
       [56., 59., 66., -1.]])

====> processing power of edge server <====
array([ 8, 12,  8,  7])

====> All simple paths between any two server <====
[[[[0]],
  [[0, 1], [0, 2, 1], [0, 2, 3, 1], [0, 3, 1], [0, 3, 2, 1]],
  [[0, 1, 2], [0, 1, 3, 2], [0, 2], [0, 3, 1, 2], [0, 3, 2]],
  [[0, 1, 2, 3], [0, 1, 3], [0, 2, 1, 3], [0, 2, 3], [0, 3]]],
 [[[1, 0], [1, 2, 0], [1, 2, 3, 0], [1, 3, 0], [1, 3, 2, 0]],
  [[1]],
  [[1, 0, 2], [1, 0, 3, 2], [1, 2], [1, 3, 0, 2], [1, 3, 2]],
  [[1, 0, 2, 3], [1, 0, 3], [1, 2, 0, 3], [1, 2, 3], [1, 3]]],
 [[[2, 0], [2, 1, 0], [2, 1, 3, 0], [2, 3, 0], [2, 3, 1, 0]],
  [[2, 0, 1], [2, 0, 3, 1], [2, 1], [2, 3, 0, 1], [2, 3, 1]],
  [[2]],
  [[2, 0, 1, 3], [2, 0, 3],

In [4]:
# 进行算法对比 DPE
dpe = DPE(G, bw, pp, simple_paths, reciprocals_list, proportions_list, pp_required, data_stream)

start = datetime.datetime.now()

cpu_earliest_finish_time_all_dpe, task_deployment_all_dpe, cpu_task_mapping_list_all_dpe, task_start_time_all_dpe = \
    dpe.get_response_time(sorted_job_path=para.get("batch_task_topological_order_path"))

print(task_deployment_all_dpe)

end = datetime.datetime.now()

print('Computer\'s running time:', (end - start).microseconds, 'microseconds')

job_num_chosen = 50  # 随机选择采样出来的 DAG 中的某一个
# show_DAG()  # TODO
scheduling_result = \
    SchedulingResult(cpu_earliest_finish_time_all_dpe, task_deployment_all_dpe, cpu_task_mapping_list_all_dpe,
                     task_start_time_all_dpe, job_num_chosen)
scheduling_result.print()


Getting makespan for 100 jobs by DPE algorithm ...
100% [##################################################]
The overall makespan achieved by DPE: 15.097876 seconds
The average makespan: 0.150979 seconds


ValueError: too many values to unpack (expected 4)

In [ ]:
# 进行算法对比 HEFT
heft = HEFT(G, bw, pp, simple_paths, reciprocals_list, proportions_list, pp_required, data_stream)
start = datetime.datetime.now()
cpu_task_mapping_list_all, task_deployment_all = heft.get_response_time(
    sorted_job_path=para.get("batch_task_topological_order_path"))
end = datetime.datetime.now()
print('Computer\'s running time:', (end - start).microseconds, 'microseconds')
print('\nThe finish time of each task on the chosen cpu for job #%d:' % job_num_chosen)
pprint.pprint(cpu_task_mapping_list_all[job_num_chosen])